# PG-MoE — 联合训练

把所有模块串起来，端到端训练：

```
Vision tokens (Qwen, cached) ──> VisionProjector ──┐
                                                    ├─ CrossAttn ─┐
Raw IMU ──> IMUExpert (pretrained, 82.33%) ────────┘             │
                                                                  ▼
                                                          GatedFusion (α 加权)
                                                                  │
Raw IMU ──> PhaseArbitrator ──> α(t)                              ▼
                                                          Classification head → 27 类
```

**前提：**
- 你的 IMU expert 已经训好，权重在 Drive `pgmoe_ckpt/imu_expert.pt`
- Hang 的 vision tokens 已缓存在 Drive `utd_mhad/vision_tokens_qwen25.pt`
- 这个 repo 已经 clone 到 Drive 上（如果没有，把 `code/` 文件夹放 Drive 任意位置，改下面的 `CODE_DIR`）

**Runtime 建议**：A100 或 V100 + High-RAM。Qwen cache 是 13.5 GB，加载需要充足 RAM。

---

## 1. 挂载 Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. 路径与超参

`CODE_DIR` 要指向 repo 里 `project/final/code` 的实际位置——这样我们能 `from models.pgmoe import PGMoE`。

In [ ]:
import os, sys

# 改成你 Drive 上 repo 的实际路径
CODE_DIR = "/content/drive/MyDrive/Multi-Modal-AI/project/final/code"

DATA_ROOT       = "/content/drive/MyDrive/utd_mhad"
VISION_CACHE    = os.path.join(DATA_ROOT, "vision_tokens_qwen25.pt")
IMU_CKPT_PATH   = "/content/drive/MyDrive/pgmoe_ckpt/imu_expert.pt"
SAVE_DIR        = "/content/drive/MyDrive/pgmoe_ckpt"

NUM_CLASSES     = 27
D_MODEL         = 256
T_I             = 12
VISION_N_FRAMES = 60

EPOCHS          = 50
BATCH_SIZE      = 16
LR              = 3e-4
WEIGHT_DECAY    = 1e-4
DROPOUT         = 0.3
FOCAL_GAMMA     = 2.0

os.makedirs(SAVE_DIR, exist_ok=True)
sys.path.insert(0, CODE_DIR)
print("CODE_DIR:", CODE_DIR)
print("VISION_CACHE exists:", os.path.exists(VISION_CACHE))
print("IMU_CKPT_PATH exists:", os.path.exists(IMU_CKPT_PATH))

## 3. 导入

我们的模块全部从 `models/` 导入。如果上面 `CODE_DIR` 没指对，下面会报 ModuleNotFoundError。

In [ ]:
import gc, time
import numpy as np
import scipy.io as sio
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, confusion_matrix
import matplotlib.pyplot as plt

from models.pgmoe import PGMoE, FocalLoss

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

## 4. 加载 Hang 的 vision token cache

**注意：cache 是 13.5 GB**，加载需要充足 RAM。加载完后我们立刻做 **spatial pooling**（64 个 patch token 平均成 1 个），把每个样本从 `(60, 64, 2048)` 压成 `(60, 2048)`，节省 64 倍内存。原 dict 用 `del` 释放。

处理后约 380 MB，CPU RAM 完全够。

In [ ]:
print("Loading Qwen vision tokens cache (13.5 GB)...")
t0 = time.time()
raw_cache = torch.load(VISION_CACHE, map_location="cpu", weights_only=False)
print(f"  loaded {len(raw_cache)} samples in {time.time()-t0:.1f}s")

first_key = next(iter(raw_cache))
first_val = raw_cache[first_key]
print(f"  sample key={first_key}, value shape={tuple(first_val.shape)}, dtype={first_val.dtype}")

print("Spatial-pooling 64 patches per frame -> compact cache...")
t0 = time.time()
vision_cache = {}
for k, v in raw_cache.items():
    vision_cache[k] = v.mean(dim=1).contiguous().to(torch.float16)
del raw_cache
gc.collect()
print(f"  done in {time.time()-t0:.1f}s, {len(vision_cache)} entries")
print(f"  per-sample shape: {tuple(next(iter(vision_cache.values())).shape)}")

## 5. 联合 Dataset

返回 `(vision_tokens, imu, label)` 三元组。

- `vision_tokens`: `(60, 2048)`（已 spatial-pool）
- `imu`: `(6, 192)`
- `label`: int 0-26

Train/test split 用 subjects {1,3,5,7} / {2,4,6,8}，**和 IMU expert 完全一致**。

In [ ]:
IMU_LEN = 192
TRAIN_SUBJECTS = {1, 3, 5, 7}
TEST_SUBJECTS  = {2, 4, 6, 8}

def load_imu(action, subject, trial):
    fname = f"a{action}_s{subject}_t{trial}_inertial.mat"
    fpath = os.path.join(DATA_ROOT, "Inertial", fname)
    if not os.path.exists(fpath):
        return None
    data = sio.loadmat(fpath)["d_iner"].astype(np.float32)
    if data.shape[0] < IMU_LEN:
        pad = np.zeros((IMU_LEN - data.shape[0], 6), np.float32)
        data = np.concatenate([data, pad], axis=0)
    return torch.from_numpy(data[:IMU_LEN]).T.contiguous()  # (6, 192)

class JointDataset(Dataset):
    def __init__(self, vision_cache, train=True):
        allowed = TRAIN_SUBJECTS if train else TEST_SUBJECTS
        self.samples = []
        for (action, subject, trial), v_tok in vision_cache.items():
            if subject not in allowed:
                continue
            imu = load_imu(action, subject, trial)
            if imu is None:
                continue
            self.samples.append((v_tok, imu, action - 1))
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        v_tok, imu, label = self.samples[idx]
        return v_tok.float(), imu, label

train_ds = JointDataset(vision_cache, train=True)
test_ds  = JointDataset(vision_cache, train=False)
print(f"Train: {len(train_ds)}  |  Test: {len(test_ds)}")

v0, i0, y0 = train_ds[0]
print(f"Sample 0: vision {tuple(v0.shape)}, imu {tuple(i0.shape)}, label {y0}")

## 6. 构造 PG-MoE 模型 + 加载 IMU 预训练权重

**关键步骤**：`load_pretrained_imu` 把我们之前训好的 82.33% IMU encoder 权重灌进 `imu_expert` 模块。其他模块（vision projector / cross-attn / phase arbitrator / fusion / head）都是从随机初始化训。

In [ ]:
model = PGMoE(
    num_classes=NUM_CLASSES,
    d_model=D_MODEL,
    T_i=T_I,
    vision_n_frames=VISION_N_FRAMES,
    vision_d_in=2048,
    vision_n_layers=2,
    attn_n_layers=2,
    attn_n_heads=4,
    dropout=DROPOUT,
).to(device)

model.load_pretrained_imu(IMU_CKPT_PATH, device=device)
print("Loaded pretrained IMU encoder from", IMU_CKPT_PATH)

n_total = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Params: {n_total:,} total ({n_trainable:,} trainable)")

## 7. Forward pass 烟雾测试

用 dataloader 取一个 batch 跑一遍 forward，确认所有 shape 对得上。如果这里报错（最常见的是 shape mismatch），后面就别训了。

In [ ]:
test_loader = DataLoader(test_ds, batch_size=4, shuffle=False)
v, i, y = next(iter(test_loader))
v, i = v.to(device), i.to(device)

model.eval()
with torch.no_grad():
    logits, alpha = model(v, i, return_alpha=True)
print("Input  vision:", tuple(v.shape))
print("Input  imu   :", tuple(i.shape))
print("Output logits:", tuple(logits.shape))   # 期望 (4, 27)
print("Output alpha :", tuple(alpha.shape))    # 期望 (4, 12)
print("Alpha range  :", float(alpha.min()), "to", float(alpha.max()))

## 8. 训练 setup：DataLoader / Optimizer / Focal Loss / Scheduler

- **Focal Loss**（γ=2）：plan 第二节模块 5 要求，对难类别加权
- **AdamW**：和 IMU 训练一致
- **Cosine LR**：和 IMU 训练一致
- **小 batch + 较低 lr**：因为预训练 IMU encoder 已经训得很好，太大 lr 会把它训坏

In [ ]:
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion = FocalLoss(gamma=FOCAL_GAMMA)

print(f"Train batches: {len(train_loader)}, Test batches: {len(test_loader)}")

## 9. 评估函数

In [ ]:
def evaluate(model, loader):
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for v, i, y in loader:
            logits = model(v.to(device), i.to(device))
            preds.extend(logits.argmax(1).cpu().numpy())
            labels.extend(y.numpy())
    return accuracy_score(labels, preds), np.array(preds), np.array(labels)

## 10. 训练循环

**预期：**
- 前几个 epoch acc 可能不高（很多模块随机初始化）
- 大概 20–30 epoch 后应该超过 IMU 单模态 82.33%
- 目标：超过 vision 单模态 ResNet3D 89.3% 是有说服力的故事
- A100 GPU 上 50 epoch 大概 10–15 分钟

每行末尾的 `*` 表示该 epoch 是当前最佳。

In [ ]:
best_acc, best_epoch = 0.0, -1
history = {"loss": [], "acc": []}

for epoch in range(1, EPOCHS + 1):
    model.train()
    t0 = time.time()
    train_loss = 0.0
    for v, i, y in train_loader:
        v, i, y = v.to(device), i.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(v, i), y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        train_loss += loss.item() * v.size(0)
    scheduler.step()
    train_loss /= len(train_ds)

    acc, _, _ = evaluate(model, test_loader)
    history["loss"].append(train_loss)
    history["acc"].append(acc)

    flag = ""
    if acc > best_acc:
        best_acc, best_epoch = acc, epoch
        torch.save(model.state_dict(), os.path.join(SAVE_DIR, "pgmoe_best.pt"))
        flag = "  *"
    print(f"Epoch {epoch:3d} | loss {train_loss:.4f} | "
          f"acc {acc:.4f} | {time.time()-t0:.1f}s{flag}")

print(f"\nBest test acc: {best_acc:.4f} ({best_acc*100:.2f}%) at epoch {best_epoch}")
print(f"  vs IMU-only      82.33%")
print(f"  vs Vision-Qwen   58.6%")
print(f"  vs ResNet3D-only 89.3%")

## 11. 训练曲线

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history["loss"]); axes[0].set_title("Train Focal Loss"); axes[0].set_xlabel("epoch"); axes[0].grid(True, alpha=0.3)
axes[1].plot(history["acc"]);  axes[1].set_title("Test Accuracy");   axes[1].set_xlabel("epoch"); axes[1].grid(True, alpha=0.3)
axes[1].axhline(0.8233, ls="--", color="red",    alpha=0.6, label="IMU 82.33%")
axes[1].axhline(0.586,  ls="--", color="orange", alpha=0.6, label="Qwen 58.6%")
axes[1].axhline(0.893,  ls="--", color="green",  alpha=0.6, label="ResNet3D 89.3%")
axes[1].legend(loc="lower right")
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "pgmoe_training_curve.png"), dpi=120)
plt.show()

## 12. 用最佳模型评估 + per-class accuracy + confusion matrix

重点对比之前 IMU 单模态最差的 3 类：
- c4  right arm throw    (IMU: 25.0%)
- c18 right hand knock   (IMU: 43.8%)
- c16 tennis serve       (IMU: 50.0%)

如果 PG-MoE 在这 3 类上有明显提升，**就直接证明了 phase arbitrator 让 vision 接管这些类的效果**。

In [ ]:
UTD_LABELS = [
    "right arm swipe left", "right arm swipe right",
    "right hand wave", "two hand front clap", "right arm throw",
    "cross arms in chest", "basketball shooting", "right hand draw x",
    "right hand draw circle CW", "right hand draw circle CCW",
    "draw triangle", "bowling", "front boxing", "baseball swing",
    "tennis right hand forehand", "arm curl", "tennis serve",
    "two hand push", "right hand knock", "right hand catch",
    "right hand pickup and throw", "jogging", "walking",
    "sit to stand", "stand to sit", "forward lunge", "squat",
]
IMU_PER_CLASS = {4: 0.250, 18: 0.438, 16: 0.500}  # midterm-ish baseline

model.load_state_dict(torch.load(os.path.join(SAVE_DIR, "pgmoe_best.pt")))
acc, preds, labels = evaluate(model, test_loader)
print(f"PG-MoE test acc: {acc:.4f} ({acc*100:.2f}%)\n")

print("Per-class accuracy (PG-MoE vs IMU baseline where known):")
rows = []
for c in range(NUM_CLASSES):
    mask = labels == c
    if mask.sum() == 0:
        continue
    rows.append((c, float((preds[mask] == c).mean()), int(mask.sum())))
rows.sort(key=lambda r: r[1])
for c, acc_c, n in rows:
    name = UTD_LABELS[c] if c < len(UTD_LABELS) else f"class {c}"
    bar = "#" * int(acc_c * 30)
    base = f" (IMU was {IMU_PER_CLASS[c]:.3f})" if c in IMU_PER_CLASS else ""
    print(f"  c{c:2d} acc {acc_c:.3f} n={n:2d} | {bar:<30} {name}{base}")

cm = confusion_matrix(labels, preds, labels=list(range(NUM_CLASSES)))
plt.figure(figsize=(8, 7))
plt.imshow(cm, cmap="Blues")
plt.colorbar(); plt.title(f"PG-MoE Confusion Matrix (acc={acc*100:.2f}%)")
plt.xlabel("predicted"); plt.ylabel("true")
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "pgmoe_confusion_matrix.png"), dpi=120)
plt.show()

## 13. 训练后的 α(t) 可视化

**Paper 的杀手图**：拿 throw / walking / squat 三个有代表性的样本，画训练后的 α(t)。

**期望看到：**
- throw 在 timestep ~7（IMU 序列中 impact 位置）α 升高（信 vision）
- walking / squat 整体 α 偏低（信 IMU）
- 不再像未训练时那样三条线重合在 0.5

In [ ]:
def get_alpha_for_action(action_id_1indexed, subject=2, trial=1):
    key = (action_id_1indexed, subject, trial)
    if key not in vision_cache:
        return None, None
    v_tok = vision_cache[key].float().unsqueeze(0).to(device)
    imu = load_imu(action_id_1indexed, subject, trial)
    if imu is None:
        return None, None
    imu = imu.unsqueeze(0).to(device)
    model.eval()
    with torch.no_grad():
        _, alpha = model(v_tok, imu, return_alpha=True)
    return alpha.squeeze().cpu().numpy(), imu.squeeze().cpu().numpy()

demo = [
    (5,  "throw",   "red"),
    (23, "walking", "blue"),
    (27, "squat",   "green"),
]

fig, ax = plt.subplots(figsize=(10, 4))
for action_id, label, color in demo:
    a, _ = get_alpha_for_action(action_id, subject=2, trial=1)
    if a is None:
        continue
    ax.plot(np.linspace(0, 1, len(a)), a, "o-", color=color, label=label, linewidth=2)
ax.axhline(0.5, ls="--", color="gray", alpha=0.5)
ax.set_ylim(0, 1)
ax.set_xlabel("normalized time within action")
ax.set_ylabel("α  (1=trust vision, 0=trust IMU)")
ax.set_title("α(t) after PG-MoE joint training")
ax.grid(True, alpha=0.3); ax.legend(loc="best")
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "alpha_trained.png"), dpi=120)
plt.show()

## 14. 打包下载

把 4 个产物打包：
- `pgmoe_best.pt` (~30 MB, 完整模型权重)
- `pgmoe_training_curve.png`
- `pgmoe_confusion_matrix.png`
- `alpha_trained.png`
- 加一份 `pgmoe_results.txt`

In [ ]:
import shutil
BUNDLE_DIR = "/content/pgmoe_archive"
os.makedirs(BUNDLE_DIR, exist_ok=True)

for src in ["pgmoe_best.pt", "pgmoe_training_curve.png",
            "pgmoe_confusion_matrix.png", "alpha_trained.png"]:
    p = os.path.join(SAVE_DIR, src)
    if os.path.exists(p):
        shutil.copy(p, os.path.join(BUNDLE_DIR, src))

lines = []
lines.append("# PG-MoE Joint Training Results\n\n")
lines.append(f"Best test accuracy: {best_acc:.4f}  ({best_acc*100:.2f}%)\n")
lines.append(f"Best epoch:         {best_epoch}\n\n")
lines.append("## Baselines\n")
lines.append(f"  IMU only:      82.33%\n")
lines.append(f"  Qwen vision:   58.60%\n")
lines.append(f"  ResNet3D:      89.30%\n")
lines.append(f"  Midterm IMU:   67.90%\n\n")
lines.append("## Per-class accuracy (sorted ascending)\n")
for c, acc_c, n in rows:
    name = UTD_LABELS[c] if c < len(UTD_LABELS) else f"class {c}"
    base = f" (IMU was {IMU_PER_CLASS[c]:.3f})" if c in IMU_PER_CLASS else ""
    lines.append(f"  c{c:2d}  acc={acc_c:.3f}  n={n:2d}  {name}{base}\n")
with open(os.path.join(BUNDLE_DIR, "pgmoe_results.txt"), "w") as f:
    f.writelines(lines)

print("Bundle contents:")
for fn in sorted(os.listdir(BUNDLE_DIR)):
    sz = os.path.getsize(os.path.join(BUNDLE_DIR, fn)) / (1024*1024)
    print(f"  {fn}  ({sz:.2f} MB)")

zip_path = "/content/pgmoe_archive.zip"
shutil.make_archive(zip_path.replace(".zip", ""), "zip", BUNDLE_DIR)
print(f"\nzipped: {zip_path}  ({os.path.getsize(zip_path)/(1024*1024):.2f} MB)")

from google.colab import files
files.download(zip_path)

## 完事

解压 `pgmoe_archive.zip`，把 4 个文件放进 repo：
- `pgmoe_best.pt` → `project/final/checkpoints/`
- 三张 png + `pgmoe_results.txt` → `project/final/figures/`

告诉 Claude 数字，他会更新 devlog 并 commit + push。